<a href="https://colab.research.google.com/github/fralfaro/MAT306/blob/main/docs/labs/lab_04.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MAT306 - Laboratorio N°04


**Objetivo**: Aplicar técnicas intermedias y avanzadas de análisis de datos con pandas utilizando un caso real: el Índice de Libertad de Prensa. Este laboratorio incluye operaciones de limpieza, transformación, combinación de datos, y análisis exploratorio usando `merge`, `groupby`, `concat` y otras funciones fundamentales.




**Descripción del Dataset**

El presente conjunto de datos está orientado al análisis del **Índice de Libertad de Prensa**, una métrica internacional que evalúa el nivel de libertad del que gozan periodistas y medios de comunicación en distintos países. Este índice es recopilado anualmente por la organización **Reporteros sin Fronteras**.

La base de datos contempla observaciones por país y año, e incluye tanto el valor del índice como el ranking correspondiente. A menor puntaje en el índice, mayor nivel de libertad de prensa.

**Diccionario de variables**

| Variable     | Clase    | Descripción                                                                          |
| ------------ | -------- | ------------------------------------------------------------------------------------ |
| `codigo_iso` | carácter | Código ISO 3166-1 alfa-3 que representa a cada país.                                 |
| `pais`       | carácter | Nombre oficial del país.                                                             |
| `anio`       | entero   | Año en que se registró la medición del índice.                                       |
| `indice`     | numérico | Valor numérico del Índice de Libertad de Prensa (menor valor indica mayor libertad). |
| `ranking`    | entero   | Posición relativa del país en el ranking mundial de libertad de prensa.              |


**Fuente original y adaptación pedagógica**

* **Fuente original**: [Reporteros sin Fronteras](https://www.rsf-es.org/), recopilado y publicado a través del portal del [Banco Mundial](https://tcdata360.worldbank.org/indicators/h3f86901f?country=BRA&indicator=32416&viz=line_chart&years=2001,2019).
* **Adaptación educativa**: Los archivos han sido modificados intencionalmente para incorporar desafíos técnicos que permiten aplicar los contenidos abordados en clases, tales como limpieza de datos, normalización, detección de duplicados, y combinación de fuentes.


**Descripción de los archivos disponibles**

* **`libertad_prensa_codigo.csv`**: Contiene los pares `codigo_iso` y `pais`. Incluye intencionalmente un código ISO con dos nombres distintos de país para efectos de limpieza y validación de datos.

* **`libertad_prensa_01.csv`**: Contiene registros de los años **anteriores a 2010**. Incluye las variables `PAIS`, `ANIO`, `INDICE`, y `RANKING` con nombres de columna en **mayúsculas**.

* **`libertad_prensa_02.csv`**: Contiene registros de los años **desde 2010 en adelante**. Estructura similar al archivo anterior, con nombres de columna también en **mayúsculas**.





In [1]:
import numpy as np
import pandas as pd

# lectura de datos
archivos_anio = [
    'https://raw.githubusercontent.com/fralfaro/MAT306/main/docs/labs/data/libertad_prensa_01.csv',
    'https://raw.githubusercontent.com/fralfaro/MAT306/main/docs/labs/data/libertad_prensa_02.csv'
 ]
df_codigos = pd.read_csv('https://raw.githubusercontent.com/fralfaro/MAT306/main/docs/labs/data/libertad_prensa_codigo.csv')



### 1. Consolidación y limpieza de datos

A partir de los archivos disponibles, realice los siguientes pasos:

**a)** Cree un DataFrame llamado `df_anio` que consolide la información proveniente de los archivos **`libertad_prensa_01.csv`** y **`libertad_prensa_02.csv`**, correspondientes a distintas ventanas de tiempo. Recuerde que ambos archivos tienen nombres de columnas en mayúscula, por lo que debe normalizarlas a **minúscula** para asegurar consistencia.

**b)** Explore el archivo **`libertad_prensa_codigo.csv`** e identifique el código ISO que aparece asociado a dos nombres de país distintos. Elimine el registro que corresponda a un valor incorrecto o inconsistente, conservando solo el que considere válido.

**c)** Una vez preparados los archivos, cree un nuevo DataFrame llamado `df` que combine `df_anio` con `df_codigos`, utilizando la columna `codigo_iso` como clave. Asegúrese de realizar una unión que conserve únicamente los registros que tengan coincidencia en ambas fuentes.

> **Sugerencia**:
>
> * Para unir los archivos por filas (años), utilice la función `pd.concat([...])`.
> * Para combinar información por columnas (variables), utilice `pd.merge(...)` especificando `on='codigo_iso'`.



In [6]:
# a)
df_anio_list = []
for url in archivos_anio:
    df = pd.read_csv(url)
    df.columns = df.columns.str.lower()
    df_anio_list.append(df)

df_anio = pd.concat(df_anio_list, ignore_index=True)


# b)
duplicated_codes = df_codigos[df_codigos.duplicated('codigo_iso', keep=False)]
print("Duplicados:")
display(duplicated_codes)

df_codigos_cleaned = df_codigos[~((df_codigos['codigo_iso'] == 'ZWE') & (df_codigos['pais'] == 'malo'))].copy()


# c)
df = pd.merge(df_anio, df_codigos_cleaned, on='codigo_iso', how='inner')

display(df.head())

Duplicados:


,codigo_iso,pais
179,ZWE,Zimbabue
180,ZWE,malo


,codigo_iso,anio,indice,ranking,pais
0,AFG,2001,35.5,59.0,Afghanistán
1,AGO,2001,30.2,50.0,Angola
2,ALB,2001,NaN,NaN,Albania
3,AND,2001,NaN,NaN,Andorra
4,ARE,2001,NaN,NaN,Emiratos Árabes Unidos




### 2. Exploración inicial del conjunto de datos

Una vez que hayas consolidado el DataFrame final `df`, realiza un análisis exploratorio básico respondiendo las siguientes preguntas:

#### **Estructura del DataFrame**

* ¿Cuántas **filas (observaciones)** contiene el conjunto de datos?
* ¿Cuántas **columnas** tiene el DataFrame?
* ¿Cuáles son los **nombres de las columnas**?
* ¿Qué **tipo de datos** tiene cada columna?
* ¿Hay columnas con un tipo de dato inesperado (por ejemplo, fechas como strings)?

#### **Resumen estadístico**

* Genera un resumen estadístico del conjunto de datos con `.describe()`.
  ¿Qué observas sobre los valores de `indice` y `ranking`?
* ¿Qué valores mínimo, máximo y promedio tiene la columna `indice`?
* ¿Qué países presentan los valores extremos en `indice` y `ranking`?

#### **Datos faltantes**

* ¿Cuántos valores nulos hay en cada columna?
* ¿Qué proporción de observaciones tienen valores faltantes?
* ¿Hay columnas con más del 30% de datos faltantes?

#### **Unicidad y duplicados**

* ¿Cuántos países distintos (`pais`) hay en el DataFrame?
* ¿Cuántos años distintos (`anio`) hay representados?
* ¿Existen filas duplicadas (exactamente iguales)? ¿Cuántas?

#### **Validación cruzada de columnas**

* ¿Hay inconsistencias entre el país (`pais`) y su código (`codigo_iso`)?
  (por ejemplo, un mismo código ISO asociado a más de un país)

> **Sugerencia**: Apoya tu análisis con funciones como `.info()`, `.nunique()`, `.isnull().sum()`, `.duplicated()`, `.value_counts()`, entre otras.



    

In [8]:
# Estructura del DataFrame
print("Estructura del DataFrame:")
df.info()

# Resumen estadístico
print("\nResumen estadístico:")
display(df.describe())
print("Los datos indice y ranking tiene una std muy grande comparado a la magnitud de sus valores")
print("el minimo, maximo y promedio son: 0, 64.536 y 28, respectivamente")
print("los valores extremos de indice y ranking tiene muchos ordenes de magnitud de diferencia, lo que podria dificultar el analisis de los datos")

# Datos faltantes
print("\nValores nulos por columna:")
print(df.isnull().sum())

print("\nProporción de valores faltantes:")
print(df.isnull().sum() / len(df))

# Unicidad y duplicados
print("\nNúmero de países distintos:", df['pais'].nunique())
print("Número de años distintos:", df['anio'].nunique())
print("Número de filas duplicadas:", df.duplicated().sum())

# Validación cruzada de columnas
print("\nInconsistencias entre pais y codigo_iso:")
inconsistent_codes = df.groupby('codigo_iso')['pais'].nunique()
inconsistent_codes = inconsistent_codes[inconsistent_codes > 1]
if len(inconsistent_codes) > 0:
  print(inconsistent_codes)
else :
  print("No hay inconsistencias entre pais y codigo_iso")

Estructura del DataFrame:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3060 entries, 0 to 3059
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   codigo_iso  3060 non-null   object 
 1   anio        3060 non-null   int64  
 2   indice      2664 non-null   float64
 3   ranking     2837 non-null   float64
 4   pais        3060 non-null   object 
dtypes: float64(2), int64(1), object(2)
memory usage: 119.7+ KB

Resumen estadístico:


,anio,indice,ranking
count,3060.000000,2664.000000,2837.000000
mean,2009.941176,205.782316,477.930913
std,5.786024,2695.525264,6474.935347
min,2001.000000,0.000000,1.000000
25%,2005.000000,15.295000,34.000000
50%,2009.000000,28.000000,70.000000
75%,2015.000000,41.227500,110.000000
max,2019.000000,64536.000000,121056.000000


Los datos indice y ranking tiene una std muy grande comparado a la magnitud de sus valores
el minimo, maximo y promedio son: 0, 64.536 y 28, respectivamente
los valores extremos de indice y ranking tiene muchos ordenes de magnitud de diferencia, lo que podria dificultar el analisis de los datos

Valores nulos por columna:
codigo_iso      0
anio            0
indice        396
ranking       223
pais            0
dtype: int64

Proporción de valores faltantes:
codigo_iso    0.000000
anio          0.000000
indice        0.129412
ranking       0.072876
pais          0.000000
dtype: float64

Número de países distintos: 179
Número de años distintos: 17
Número de filas duplicadas: 0

Inconsistencias entre pais y codigo_iso:
No hay inconsistencias entre pais y codigo_iso





### 3. Comparación regional: países latinoamericanos

En esta sección se busca identificar cuáles son los países de América Latina que han presentado los valores extremos del **Índice de Libertad de Prensa** en cada año observado.

> Recuerda que un menor puntaje en `indice` implica mayor libertad de prensa.

#### **Tareas:**

**a)** Utilizando un ciclo `for`, recorre cada año del conjunto de datos filtrado por países latinoamericanos, y determina para cada año:

* El país con el menor valor de `indice` (mayor libertad de prensa).
* El país con el mayor valor de `indice` (menor libertad de prensa).

**b)** Resuelve la misma tarea del punto anterior utilizando un enfoque vectorizado con `groupby`, sin usar ciclos explícitos.



#### **Lista de países latinoamericanos considerada:**

```python
df_america = ['ARG', 'ATG', 'BLZ', 'BOL', 'BRA', 'CAN', 'CHL', 'COL', 'CRI',
           'CUB', 'DOM', 'ECU', 'GRD', 'GTM', 'GUY', 'HND', 'HTI', 'JAM',
           'MEX', 'NIC', 'PAN', 'PER', 'PRY', 'SLV', 'SUR', 'TTO', 'URY',
           'USA', 'VEN']
```

> Puedes usar esta lista para filtrar el DataFrame final por la columna `codigo_iso`.



In [16]:
# Filter df for Latin American countries
america = ['ARG', 'ATG', 'BLZ', 'BOL', 'BRA', 'CAN', 'CHL', 'COL', 'CRI',
       'CUB', 'DOM', 'ECU', 'GRD', 'GTM', 'GUY', 'HND', 'HTI', 'JAM',
       'MEX', 'NIC', 'PAN', 'PER', 'PRY', 'SLV', 'SUR', 'TTO', 'URY',
       'USA', 'VEN']

df_america = df[df['codigo_iso'].isin(america)].copy()

# a) Using a for loop
print("Analisis usando Loop:")
for year in df_america['anio'].unique():
    df_year = df_america[df_america['anio'] == year].dropna(subset=['indice'])

    if not df_year.empty:
        min_indice_row = df_year.loc[df_year['indice'].idxmin()]
        max_indice_row = df_year.loc[df_year['indice'].idxmax()]

        print(f"\nAño: {year}")
        print(f"  Pais con menor indice de libertad (mas libre): {min_indice_row['pais']} (Index: {min_indice_row['indice']:.2f})")
        print(f"  Pais con mayor indice de libertad (menos libre): {max_indice_row['pais']} (Index: {max_indice_row['indice']:.2f})")
    else:
        print(f"\nAño: {year} - error en los datos analizados.")


# b) Using groupby
print("\nAnalisis usando groupby:")
def get_extreme_countries(group):
    if group['indice'].isnull().all():
        return pd.Series([None, None, None, None], index=['min_pais', 'min_indice', 'max_pais', 'max_indice'])
    min_row = group.loc[group['indice'].idxmin()]
    max_row = group.loc[group['indice'].idxmax()]
    return pd.Series([min_row['pais'], min_row['indice'], max_row['pais'], max_row['indice']],
                     index=['min_pais', 'min_indice', 'max_pais', 'max_indice'])

extreme_countries_by_year = df_america.groupby('anio').apply(get_extreme_countries)
display(extreme_countries_by_year)

Analisis usando Loop:

Año: 2001
  Pais con menor indice de libertad (mas libre): Canadá (Index: 0.80)
  Pais con mayor indice de libertad (menos libre): Cuba (Index: 90.30)

Año: 2002
  Pais con menor indice de libertad (mas libre): Trinidad y Tobago (Index: 1.00)
  Pais con mayor indice de libertad (menos libre): Cuba (Index: 97.83)

Año: 2003
  Pais con menor indice de libertad (mas libre): Trinidad y Tobago (Index: 2.00)
  Pais con mayor indice de libertad (menos libre): Argentina (Index: 35826.00)

Año: 2004
  Pais con menor indice de libertad (mas libre): Trinidad y Tobago (Index: 2.00)
  Pais con mayor indice de libertad (menos libre): Cuba (Index: 87.00)

Año: 2005
  Pais con menor indice de libertad (mas libre): Bolivia (Index: 4.50)
  Pais con mayor indice de libertad (menos libre): Cuba (Index: 95.00)

Año: 2006
  Pais con menor indice de libertad (mas libre): Canadá (Index: 4.88)
  Pais con mayor indice de libertad (menos libre): Cuba (Index: 96.17)

Año: 2007
  Pais con me

/tmp/ipython-input-4152606845.py:35: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  extreme_countries_by_year = df_america.groupby('anio').apply(get_extreme_countries)


,min_pais,min_indice,max_pais,max_indice
anio,,,,
2001,Canadá,0.80,Cuba,90.30
2002,Trinidad y Tobago,1.00,Cuba,97.83
2003,Trinidad y Tobago,2.00,Argentina,35826.00
2004,Trinidad y Tobago,2.00,Cuba,87.00
2005,Bolivia,4.50,Cuba,95.00
2006,Canadá,4.88,Cuba,96.17
2007,Canadá,3.33,Cuba,88.33
2008,Canadá,3.70,Cuba,94.00
2009,Estados Unidos,6.75,Cuba,78.00


### 4. Análisis anual del índice por país

En esta sección se busca analizar la evolución del **índice máximo** de libertad de prensa alcanzado por cada país a lo largo del tiempo.

#### **Tarea principal:**

* Construye una tabla dinámica (`pivot_table`) donde las **filas** correspondan a los países, las **columnas** a los años (`anio`) y los **valores** sean el `indice` máximo alcanzado por cada país en ese año.
* Asegúrate de reemplazar los valores nulos resultantes con `0`.

> **Hint**: Puedes utilizar el parámetro `fill_value=0` en `pd.pivot_table(...)`.



#### **Preguntas adicionales:**

**a)** ¿Qué país tiene el mayor valor de `indice` en toda la tabla resultante? ¿Y cuál tiene el menor (distinto de cero)?
**b)** ¿Qué años presentan en promedio los valores de `indice` más altos? ¿Y los más bajos?

> (Pista: usa `.mean(axis=0)` sobre la tabla pivot)

**c)** ¿Qué país muestra mayor **variabilidad** (diferencia entre su máximo y mínimo `indice` a lo largo del tiempo)?

> (Pista: aplica `.max(axis=1) - .min(axis=1)`)

**d)** ¿Existen países con índice constante a lo largo de todos los años registrados? ¿Cuáles?

**e)** ¿Qué países no tienen ningún dato (es decir, quedaron con todos los valores igual a 0)? ¿Podrías explicar por qué?





In [19]:
pivot_table = pd.pivot_table(df, values='indice', index='pais', columns='anio', aggfunc='max', fill_value=0)
display(pivot_table)

# a)
max_indice_country = pivot_table.max().max()
min_indice_country = pivot_table[pivot_table > 0].min().min()

country_max = pivot_table[pivot_table == max_indice_country].stack().idxmax()[0]
country_min = pivot_table[pivot_table == min_indice_country].stack().idxmin()[0]

print(f"\nPais con el mayor valor de indice en toda la tabla resultante: {country_max} ({max_indice_country})")
print(f"Pais con el menor valor de indice en toda la tabla resultante: {country_min} ({min_indice_country})")

# b)
avg_indice_by_year = pivot_table.mean(axis=0)
year_highest_avg = avg_indice_by_year.idxmax()
year_lowest_avg = avg_indice_by_year.idxmin()

print(f"\nAño con el mayor indice promedio: {year_highest_avg} ({avg_indice_by_year.max():.2f})")
print(f"Año con el menor indice promedio: {year_lowest_avg} ({avg_indice_by_year.min():.2f})")

# c) Country with the highest variability in index
index_variability = pivot_table.max(axis=1) - pivot_table.min(axis=1)
country_highest_variability = index_variability.idxmax()

print(f"\nPais con la mayor variacion de indice: {country_highest_variability} ({index_variability.max():.2f})")

# d) Countries with constant index over the years
constant_index_countries = index_variability[index_variability == 0].index.tolist()
print(f"\nPais con indice constante: {constant_index_countries}")

# e) Countries with no data (all values are 0 in the pivot table)
countries_with_no_data = pivot_table[(pivot_table == 0).all(axis=1)].index.tolist()
print(f"\nPaises sin informacion (todo 0's en la tabla): {countries_with_no_data}")
print("Estos paises podrian no tener disponible esta informacion.")

anio,2001,2002,2003,2004,2005,2006,2007,2008,2009,2012,2013,2014,2015,2017,2018,2019
pais,,,,,,,,,,,,,,,,
Afghanistán,35.5,40.17,28.25,39.17,44.25,56.50,59.25,54.25,51.67,37.36,37.07,37.44,37.75,39.46,37.28,36.55
Albania,0.0,6.50,11.50,14.17,18.00,25.50,16.00,21.75,21.50,30.88,29.92,28.77,29.92,29.92,29.49,29.84
Alemania,1.5,1.33,2.00,4.00,5.50,5.75,4.50,3.50,4.25,10.24,10.23,11.47,14.80,14.97,14.39,14.60
Algeria,31.0,33.00,43.50,40.33,40.00,40.50,31.33,49.56,47.33,36.54,36.26,36.63,41.69,42.83,43.13,45.75
Andorra,0.0,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,6.82,6.82,19.87,19.87,21.03,22.21,24.63
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Vietnam,81.3,89.17,86.88,73.25,67.25,79.25,86.17,81.67,75.75,71.78,72.36,72.63,74.27,73.96,75.05,74.93
West Bank y Gaza,0.0,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,42.90,42.96,44.68
Yemen,34.8,41.83,48.00,46.25,54.00,56.67,59.00,83.38,82.13,69.22,67.26,66.36,67.07,65.80,62.23,61.66



Pais con el mayor valor de indice en toda la tabla resultante: Kosovo (64536.0)
Pais con el menor valor de indice en toda la tabla resultante: Austria (0.5)

Año con el mayor indice promedio: 2013 (449.11)
Año con el menor indice promedio: 2001 (20.03)

Pais con la mayor variacion de indice: Kosovo (64536.00)

Pais con indice constante: []

Paises sin informacion (todo 0's en la tabla): []
Estos paises podrian no tener disponible esta informacion.
